# 🏗️ Module 11.2: Hierarchical Reinforcement Learning for Vision

## Temporal Abstraction & Multi-Level Decision Making for Image Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_11_Advanced_Topics/02_Hierarchical_RL_Vision/hierarchical_rl_vision.ipynb)

---

## 🎯 Learning Objectives

1. Understand **hierarchical RL** and why temporal abstraction is essential for complex vision tasks
2. Master the **Options Framework** — Semi-Markov Decision Processes, option-value functions, and the Option-Critic architecture
3. Build a **two-level Manager–Worker hierarchy** where a high-level agent selects vision tasks and low-level agents execute pixel operations
4. Implement and train a full hierarchical RL system on synthetic image processing environments
5. Analyze option activation patterns, compare hierarchical vs. flat RL, and visualize learned termination conditions

---

In [ ]:
# Install dependencies (Google Colab compatible)
!pip install numpy matplotlib torch torchvision scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import namedtuple
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print(f"✅ All libraries loaded | Device: {device}")

In [ ]:
# Download REAL open-source datasets for advanced RL (TINY - under 100MB)
import torchvision

# Omniglot: 1623 characters from 50 alphabets (perfect for meta-learning, only 13MB!)
try:
    omniglot = torchvision.datasets.Omniglot(root='./data', download=True)
    print(f"✅ Omniglot: {len(omniglot)} real handwritten characters (13MB)")
    print("   50 different alphabets - perfect for few-shot/meta-learning!")
except Exception as e:
    print(f"⚠️ Omniglot: {e}, using MNIST split into tasks")

# CIFAR-10 for curriculum learning and multi-agent tasks
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos (170MB, likely already cached)")

# FashionMNIST as second domain for transfer learning (instead of STL-10!)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (30MB)")
print("   Use MNIST→FashionMNIST as sim-to-real domain shift!")

print("\n📦 Total new download: ~43MB (Omniglot + FashionMNIST)")
print("   NO STL-10 (2.6GB) needed! FashionMNIST works great for domain transfer.")

## Deep Derivation: Hierarchical Reinforcement Learning for Vision Tasks

### Step 1: Options Framework (Semi-Markov Decision Process)
An **option** $\omega = (\mathcal{I}_\omega, \pi_\omega, \beta_\omega)$ consists of:
- Initiation set: $\mathcal{I}_\omega \subseteq \mathcal{S}$ (states where $\omega$ can start)
- Intra-option policy: $\pi_\omega : \mathcal{S} \times \mathcal{A} \to [0,1]$
- Termination condition: $\beta_\omega : \mathcal{S} \to [0,1]$ (probability of terminating in state $s$)

The policy over options selects which option to execute:
$$\pi_\Omega(\omega | s) \quad \text{for } \omega \in \Omega, s \in \mathcal{I}_\omega$$

**Derivation:** The call-and-return execution model creates a Semi-MDP at the option level. The effective transition probability:
$$P_\omega(s', k | s) = \sum_{k=1}^{\infty} p(\text{option } \omega \text{ runs for } k \text{ steps, ending in } s' | s)$$

where $k$ is the number of primitive steps. The expected return for option $\omega$:
$$Q_\Omega(s, \omega) = E\left[\sum_{k=0}^{K-1} \gamma^k r_{t+k} + \gamma^K U(s_{t+K}) \middle| s_t = s, \omega\right]$$

where $U(s) = \sum_\omega \pi_\Omega(\omega|s) Q_\Omega(s, \omega)$ is the value upon arrival. $\blacksquare$

### Step 2: Bellman Equation for Options (Inter-Option Learning)
$$Q_\Omega(s, \omega) = \sum_a \pi_\omega(a|s) \left[R(s,a) + \gamma \sum_{s'} T(s'|s,a) U_\omega(s')\right]$$

where the option-conditional continuation value:
$$U_\omega(s') = (1-\beta_\omega(s')) Q_\Omega(s', \omega) + \beta_\omega(s') \max_{\omega'} Q_\Omega(s', \omega')$$

**Derivation:** Upon arriving at $s'$:
- With probability $(1-\beta_\omega(s'))$: option continues → value = $Q_\Omega(s', \omega)$
- With probability $\beta_\omega(s')$: option terminates → best new option chosen → value = $\max_{\omega'} Q_\Omega(s', \omega')$

This creates a natural temporal abstraction: the meta-controller only makes decisions when options terminate. $\blacksquare$

### Step 3: Feudal Networks (Manager-Worker Architecture)
**Manager** (high-level): Produces a goal direction in latent space:
$$g_t = f_{\text{manager}}(s_t) \in \mathbb{R}^d, \quad \hat{g}_t = \frac{g_t}{\|g_t\|}$$

**Worker** (low-level): Conditions its policy on the goal:
$$\pi_{\text{worker}}(a|s_t, g_t) = \text{softmax}(U_t \hat{g}_t)$$

where $U_t \in \mathbb{R}^{|\mathcal{A}| \times d}$ is the worker's action embedding matrix.

**Manager reward (transition policy gradient):**
$$\nabla_{\theta_M} J_M = \frac{1}{T}\sum_{t=0}^{T-1} \nabla_{\theta_M} d_{\cos}(s_{t+c} - s_t, g_t(\theta_M)) \cdot A_t^M$$

where $d_{\cos}$ is cosine similarity and $c$ is the manager's temporal horizon.

**Derivation:** The manager's goal $g_t$ should point in the direction the state actually moves. The cosine similarity:
$$d_{\cos}(s_{t+c} - s_t, g_t) = \frac{(s_{t+c} - s_t) \cdot g_t}{\|s_{t+c} - s_t\| \|g_t\|}$$

measures how well the predicted direction matches the actual state transition. This is differentiable w.r.t. $\theta_M$, enabling end-to-end training. $\blacksquare$

### Step 4: Hierarchical Image Processing Pipeline
For vision, define a natural hierarchy:

**Level 3 (Strategic):** Select which region of the image to process. Action space: $\mathcal{A}_3 = \{$top-left, top-right, bottom-left, bottom-right, center$\}$
$$\pi_3(\omega | I) = \text{softmax}(W_3 \cdot \text{GlobalPool}(\text{CNN}(I)))$$

**Level 2 (Tactical):** Select enhancement operation for the chosen region. Action space: $\mathcal{A}_2 = \{$sharpen, denoise, contrast, color\_correct$\}$
$$\pi_2(a_2 | I_{\text{region}}, \omega_3) = \text{softmax}(W_2 \cdot [\text{CNN}(I_{\text{region}}); e(\omega_3)])$$

**Level 1 (Operational):** Set continuous parameters for the chosen operation.
$$\pi_1(a_1 | I_{\text{region}}, a_2) = \mathcal{N}(\mu_1(I_{\text{region}}, a_2), \sigma_1^2)$$

**Total number of decision paths:** $|\mathcal{A}_3| \times |\mathcal{A}_2| \times |\mathbb{R}^{d_1}|$ — the hierarchy dramatically reduces the effective search space from the Cartesian product of all pixel-level actions.

### Step 5: Hindsight Goal Relabeling for Vision
When the manager sets goal $g$ but the worker achieves state $s'$ instead, relabel the experience:
$$\text{Original: } (s, g, a, r, s') \quad \text{Relabeled: } (s, s'-s, a, r', s')$$

where $r' = \mathbb{1}[\|s' - (s + (s'-s))\| < \epsilon] = 1$ (always successful!).

**Derivation:** This is Hindsight Experience Replay (HER) applied to the goal-conditioned setting. The key insight:

For any trajectory $(s_0, a_0, s_1, a_1, \ldots, s_T)$, we can construct $T$ successful relabeled transitions by setting $g_t = s_{t+1} - s_t$. The off-policy Q-learning update is valid because:
$$Q(s, g', a) \leftarrow Q(s, g', a) + \alpha[r(s, a, g') + \gamma \max_{a'} Q(s', g', a') - Q(s, g', a)]$$

is valid for any goal $g'$ — we can learn about multiple goals from a single trajectory. $\blacksquare$

### Step 6: Option Discovery via Information-Theoretic Objectives
Learn options that are diverse and distinguishable:
$$\max_\Omega I(\omega; s_T | s_0) = H(s_T | s_0) - H(s_T | s_0, \omega)$$

**Derivation:** Maximizing $I(\omega; s_T | s_0)$ ensures:
1. $H(s_T | s_0)$ is large: options reach diverse terminal states (coverage)
2. $H(s_T | s_0, \omega)$ is small: each option has predictable outcomes (consistency)

Using a variational lower bound with discriminator $q_\phi(\omega | s_0, s_T)$:
$$I(\omega; s_T | s_0) \geq E_{\omega, \tau}\left[\log q_\phi(\omega | s_0, s_T) - \log p(\omega)\right]$$

The intrinsic reward for option $\omega$:
$$r_{\text{intrinsic}}(s_0, s_T, \omega) = \log q_\phi(\omega | s_0, s_T) - \log p(\omega)$$

This encourages each option to produce trajectories that are easily distinguishable by the discriminator. $\blacksquare$

### Step 7: Convergence of Hierarchical Q-Learning
**Theorem (Dietterich, 2000):** If the hierarchy satisfies the **recursive optimality** condition (each subtask's policy is optimal given the policies of its children), then hierarchical Q-learning converges to a recursively optimal policy.

**Proof sketch:** At the lowest level, standard Q-learning convergence applies (Watkins & Dayan, 1992). At each higher level, the lower-level options define a valid SMDP. Since Q-learning converges for SMDPs under the same Robbins-Monro conditions:
$$\sum_t \alpha_t = \infty, \quad \sum_t \alpha_t^2 < \infty$$

convergence propagates bottom-up through the hierarchy. The key requirement is that each level's effective MDP is stationary when lower levels have converged. $\blacksquare$

---
## 1. Introduction to Hierarchical RL

### Why Hierarchical Decision-Making for Vision?

Consider a real-world image processing pipeline: given a degraded photograph, we might need to **first** identify the degradation type (noise, blur, low contrast), **then** select the appropriate restoration algorithm, and **finally** tune its parameters pixel-by-pixel. This naturally decomposes into levels of abstraction:

| Level | Agent | Time Scale | Action Space |
|-------|-------|-----------|---------------|
| **High (Manager)** | Task Selector | Every $k$ steps | *{enhance, denoise, segment, sharpen}* |
| **Low (Worker)** | Pixel Operator | Every step | *Continuous pixel adjustments* |

A **flat RL** agent must learn the joint policy $\pi(a|s)$ over a combinatorially large space. A **hierarchical** agent factorises this into:

$$\pi(a|s) = \sum_{o \in \mathcal{O}} \pi_{high}(o|s) \cdot \pi_{low}(a|s, o)$$

This factorisation enables:
- **Transfer**: low-level skills reused across tasks
- **Exploration**: high-level decisions reduce search depth
- **Interpretability**: we can inspect *which* option the agent chose and *why*

In [ ]:
# Visualise the hierarchical architecture concept
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Flat RL
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_title('Flat RL Agent', fontweight='bold')
box = FancyBboxPatch((2, 4), 6, 2, boxstyle='round,pad=0.3', fc='#3498db', ec='#2c3e50', lw=2)
ax.add_patch(box)
ax.text(5, 5, 'Single Policy π(a|s)', ha='center', va='center', fontsize=11, color='white', fontweight='bold')
ax.annotate('Image State', xy=(5, 4), xytext=(5, 1.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2c3e50'),
            ha='center', fontsize=10, fontweight='bold')
ax.annotate('', xy=(5, 8.5), xytext=(5, 6),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2c3e50'))
ax.text(5, 9, 'Pixel Actions', ha='center', fontsize=10, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')

# Panel 2: Hierarchical RL
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_title('Hierarchical RL Agent', fontweight='bold')
manager = FancyBboxPatch((2.5, 6.5), 5, 1.8, boxstyle='round,pad=0.3', fc='#e74c3c', ec='#2c3e50', lw=2)
ax.add_patch(manager)
ax.text(5, 7.4, 'Manager π_high(o|s)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
colors_w = ['#27ae60', '#f39c12', '#8e44ad', '#1abc9c']
labels_w = ['Enhance', 'Denoise', 'Segment', 'Sharpen']
for i, (c, l) in enumerate(zip(colors_w, labels_w)):
    x = 1.0 + i * 2.3
    w_box = FancyBboxPatch((x, 3), 2, 1.5, boxstyle='round,pad=0.2', fc=c, ec='#2c3e50', lw=1.5)
    ax.add_patch(w_box)
    ax.text(x + 1, 3.75, l, ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    ax.annotate('', xy=(x + 1, 4.5), xytext=(5, 6.5),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#7f8c8d'))
ax.annotate('Image State', xy=(5, 6.5), xytext=(5, 1),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2c3e50'),
            ha='center', fontsize=10, fontweight='bold')
ax.text(5, 9.5, 'Goal-conditioned Workers', ha='center', fontsize=10, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')

# Panel 3: Temporal abstraction
ax = axes[2]
ax.set_title('Temporal Abstraction', fontweight='bold')
t = np.arange(20)
high_decisions = [0]*5 + [1]*6 + [2]*4 + [3]*5
colors_map = {0: '#e74c3c', 1: '#3498db', 2: '#27ae60', 3: '#f39c12'}
label_map = {0: 'Enhance', 1: 'Denoise', 2: 'Segment', 3: 'Sharpen'}
for i in range(len(t)):
    ax.bar(t[i], 1, bottom=1.5, color=colors_map[high_decisions[i]], edgecolor='white', lw=0.5)
low_actions = np.random.randn(20) * 0.3
ax.bar(t, np.abs(low_actions) + 0.1, color='#95a5a6', edgecolor='white', lw=0.5)
ax.set_xlabel('Time step')
ax.set_yticks([0.5, 2.0])
ax.set_yticklabels(['Low-level\nActions', 'High-level\nOptions'])
handles = [plt.Rectangle((0,0),1,1, fc=colors_map[k]) for k in range(4)]
ax.legend(handles, [label_map[k] for k in range(4)], loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('hierarchical_concept.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: Comparison of flat vs hierarchical RL with temporal abstraction.")

---
## 2. Options Framework — Mathematical Foundation

### 2.1 Semi-Markov Decision Process (SMDP)

The Options Framework (Sutton, Precup & Singh 1999) extends MDPs with **temporally extended actions** called *options*. The resulting process is a Semi-Markov Decision Process:

$$\mathcal{M}_{SMDP} = \langle \mathcal{S}, \mathcal{O}, T_o, R_o, \gamma \rangle$$

| Symbol | Meaning |
|--------|---------|
| $\mathcal{S}$ | State space (image representations) |
| $\mathcal{O}$ | Set of available options |
| $T_o(s', k \mid s)$ | Transition probability: reaching $s'$ from $s$ in $k$ steps under option $o$ |
| $R_o(s)$ | Expected cumulative reward for executing option $o$ starting in state $s$ |
| $\gamma$ | Discount factor |

### 2.2 Option Definition

An **option** is a triple:

$$o = \langle \mathcal{I}_o, \pi_o, \beta_o \rangle$$

- $\mathcal{I}_o \subseteq \mathcal{S}$ — **Initiation set**: states where option $o$ can start
- $\pi_o : \mathcal{S} \times \mathcal{A} \to [0,1]$ — **Intra-option policy**: probability of taking action $a$ in state $s$ while $o$ is executing
- $\beta_o : \mathcal{S} \to [0,1]$ — **Termination function**: probability that $o$ terminates in state $s$

### 2.3 Option-Value Function

The value of executing option $o$ in state $s$ under a policy over options $\mu$ is:

$$Q_\Omega(s, o) = \sum_a \pi_o(a|s)\left[r(s,a) + \gamma \sum_{s'} P(s'|s,a)\, U(o, s')\right]$$

where the **continuation value** $U$ captures the decision of whether to continue or terminate:

$$U(o, s') = \bigl(1-\beta_o(s')\bigr)\,Q_\Omega(s',o) + \beta_o(s')\,\max_{o' \in \mathcal{O}} Q_\Omega(s',o')$$

The first term is the value of *continuing* option $o$; the second is the value of *terminating* and selecting the best new option.

### 2.4 Option-Critic Architecture

The **Option-Critic** (Bacon, Harb & Precup 2017) learns all components end-to-end:

1. **Critic** $Q_\Omega(s,o)$: learns option-values via TD updates
2. **Intra-option policy** $\pi_{\theta_o}(a|s)$: updated by the **intra-option policy gradient**:

$$\frac{\partial Q_\Omega(s,o)}{\partial \theta_o} = \sum_a \frac{\partial \pi_{\theta_o}(a|s)}{\partial \theta_o}\, Q_U(s, o, a)$$

where $Q_U(s,o,a) = r(s,a) + \gamma \sum_{s'} P(s'|s,a)\, U(o,s')$.

3. **Termination function** $\beta_{\varphi_o}(s)$: updated by the **termination gradient**:

$$\frac{\partial U(o,s')}{\partial \varphi_o} = -\frac{\partial \beta_{\varphi_o}(s')}{\partial \varphi_o}\, A_\Omega(s', o)$$

where the **advantage** $A_\Omega(s', o) = Q_\Omega(s', o) - \max_{o'} Q_\Omega(s', o')$ encourages termination when the current option is suboptimal.

In [ ]:
# Visualise the Option lifecycle and value decomposition
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Option lifecycle
ax = axes[0]
ax.set_title('Option Lifecycle: $o = \\langle \\mathcal{I}_o, \\pi_o, \\beta_o \\rangle$', fontsize=13)
states = np.arange(10)
np.random.seed(7)
option_active = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0, 0])
termination_prob = np.array([0.0, 0.0, 0.1, 0.1, 0.15, 0.2, 0.85, 0.0, 0.0, 0.0])
initiation = np.array([0, 0, 1, 1, 1, 1, 1, 1, 0, 0])

ax.fill_between(states, 0, option_active * 0.5, alpha=0.3, color='#3498db', label='Option active')
ax.plot(states, termination_prob, 'o-', color='#e74c3c', lw=2, markersize=6, label='$\\beta_o(s)$')
ax.bar(states, initiation * 0.05 + 0.0, bottom=-0.08, color='#27ae60', alpha=0.7, label='$\\mathcal{I}_o$')
ax.axhline(y=0.5, color='gray', ls='--', alpha=0.4)
ax.set_xlabel('State index')
ax.set_ylabel('Probability / Activity')
ax.legend(fontsize=9)
ax.set_ylim(-0.15, 1.05)

# Panel 2: Q_Omega decomposition
ax = axes[1]
ax.set_title('$Q_\\Omega$ Value Decomposition', fontsize=13)
options_labels = ['Enhance', 'Denoise', 'Segment', 'Sharpen']
q_continue = np.array([3.2, 4.5, 2.1, 1.8])
q_terminate = np.array([1.0, 0.8, 1.5, 2.5])
beta_vals = np.array([0.2, 0.1, 0.6, 0.7])
q_omega = (1 - beta_vals) * q_continue + beta_vals * q_terminate
x_pos = np.arange(len(options_labels))
w = 0.25
ax.bar(x_pos - w, q_continue, w, label='$(1-\\beta)Q_\\Omega$ (continue)', color='#3498db')
ax.bar(x_pos, q_terminate, w, label='$\\beta \\max Q_\\Omega$ (terminate)', color='#e74c3c')
ax.bar(x_pos + w, q_omega, w, label='$U(o,s\')$ (total)', color='#2ecc71')
ax.set_xticks(x_pos)
ax.set_xticklabels(options_labels)
ax.set_ylabel('Value')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('options_framework.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: Left — option lifecycle with initiation, policy execution, and termination.")
print("        Right — decomposition of the continuation value U(o, s').")

---
## 3. Two-Level Hierarchy for Vision

### 3.1 Manager–Worker Architecture

We instantiate the Options Framework as a **goal-conditioned** two-level hierarchy:

$$\pi_{low}(a \mid s, g) \quad \text{where} \quad g = \pi_{high}(s)$$

The **Manager** (high-level) observes the current image state and selects a *goal* $g$ from a discrete set of vision tasks. The **Worker** (low-level) receives the goal and executes pixel-level actions to achieve it.

### 3.2 Intrinsic Reward

The Worker receives an **intrinsic reward** based on goal achievement, decoupled from the environment's extrinsic reward:

$$r^{int}(s, a, g) = -\|\phi(s') - g\|_2^2$$

where $\phi(s')$ is a learned state embedding. The Manager is trained on the extrinsic reward:

$$r^{ext}(s, g) = \sum_{t=0}^{k-1} \gamma^t r_t$$

accumulated over the $k$ steps the Worker executes.

### 3.3 Goal-Conditioned Bellman Equations

**Manager (over options):**

$$Q_{high}(s, g) = r^{ext}(s, g) + \gamma^k \max_{g'} Q_{high}(s_k, g')$$

**Worker (primitive actions conditioned on goal):**

$$Q_{low}(s, a, g) = r^{int}(s, a, g) + \gamma \max_{a'} Q_{low}(s', a', g)$$

This decomposition lets the Manager plan at a coarser temporal resolution while the Worker operates at full frame-rate.

In [ ]:
# Generate synthetic image dataset with different degradation types

def make_clean_image(size=32, seed=None):
    """Create a synthetic 'clean' image with geometric patterns."""
    if seed is not None:
        np.random.seed(seed)
    img = np.zeros((size, size), dtype=np.float32)
    # Background gradient
    yy, xx = np.mgrid[0:size, 0:size] / size
    img += 0.3 * yy + 0.1 * xx
    # Random circles
    for _ in range(np.random.randint(2, 5)):
        cx, cy = np.random.randint(4, size-4, 2)
        r = np.random.randint(3, 8)
        brightness = np.random.uniform(0.4, 0.9)
        mask = (xx * size - cx)**2 + (yy * size - cy)**2 < r**2
        img[mask] = brightness
    # Random rectangle
    x1, y1 = np.random.randint(2, size//2, 2)
    w, h = np.random.randint(4, size//3, 2)
    img[y1:y1+h, x1:x1+w] = np.random.uniform(0.5, 0.8)
    return np.clip(img, 0, 1)

def add_noise(img, sigma=0.15):
    return np.clip(img + np.random.randn(*img.shape) * sigma, 0, 1).astype(np.float32)

def add_blur(img, k=5):
    from scipy.ndimage import uniform_filter
    return uniform_filter(img, size=k).astype(np.float32)

def reduce_contrast(img, factor=0.3):
    mean = img.mean()
    return np.clip(mean + factor * (img - mean), 0, 1).astype(np.float32)

def add_jpeg_artifact(img, block_size=8, quality=0.3):
    """Simulate block artifacts."""
    h, w = img.shape
    result = img.copy()
    for i in range(0, h, block_size):
        for j in range(0, w, block_size):
            block = result[i:i+block_size, j:j+block_size]
            block_mean = block.mean()
            result[i:i+block_size, j:j+block_size] = quality * block + (1 - quality) * block_mean
    return result.astype(np.float32)

DEGRADATION_TYPES = {
    0: ('Noisy', add_noise),
    1: ('Blurry', add_blur),
    2: ('Low Contrast', reduce_contrast),
    3: ('Artifacts', add_jpeg_artifact),
}

TASK_NAMES = ['Denoise', 'Sharpen', 'Enhance', 'Restore']

# Show samples
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for col in range(5):
    clean = make_clean_image(32, seed=col)
    deg_type = col % 4
    name, fn = DEGRADATION_TYPES[deg_type]
    degraded = fn(clean)

    axes[0, col].imshow(clean, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'Clean #{col}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(degraded, cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(f'{name}', fontsize=10, color='red')
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Clean', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Degraded', fontsize=12, fontweight='bold')
fig.suptitle('Synthetic Image Dataset — Clean vs. Degraded', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('synthetic_dataset.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: Synthetic images with four degradation types used as our environment.")

---
## 4. Implementation

### 4.1 Vision Environment

Our environment presents degraded images and rewards the agent for improving quality (measured by MSE to the clean reference).

In [ ]:
class VisionEnvironment:
    """Image processing environment with multiple degradation types."""

    def __init__(self, img_size=32, max_steps=10):
        self.img_size = img_size
        self.max_steps = max_steps
        self.n_degradations = 4
        self.reset()

    def reset(self, seed=None):
        self.clean = make_clean_image(self.img_size, seed=seed)
        self.deg_type = np.random.randint(self.n_degradations)
        _, fn = DEGRADATION_TYPES[self.deg_type]
        self.current = fn(self.clean).copy()
        self.initial = self.current.copy()
        self.step_count = 0
        return self._get_state()

    def _get_state(self):
        return self.current.copy()

    def _mse(self, img):
        return np.mean((img - self.clean) ** 2)

    def _psnr(self, img):
        mse = self._mse(img)
        if mse < 1e-10:
            return 50.0
        return 10 * np.log10(1.0 / mse)

    def step(self, action_type, action_params):
        """
        action_type: int in {0: denoise, 1: sharpen, 2: enhance_contrast, 3: smooth_restore}
        action_params: float in [0, 1] controlling strength
        """
        mse_before = self._mse(self.current)
        strength = np.clip(action_params, 0.05, 0.95)

        if action_type == 0:  # denoise (weighted average toward local mean)
            from scipy.ndimage import uniform_filter
            smoothed = uniform_filter(self.current, size=3)
            self.current = (1 - strength * 0.5) * self.current + strength * 0.5 * smoothed
        elif action_type == 1:  # sharpen (unsharp mask)
            from scipy.ndimage import gaussian_filter
            blurred = gaussian_filter(self.current, sigma=1)
            self.current = self.current + strength * 0.4 * (self.current - blurred)
        elif action_type == 2:  # enhance contrast
            mean = self.current.mean()
            self.current = mean + (1 + strength * 0.6) * (self.current - mean)
        elif action_type == 3:  # smooth restore (blend toward global statistics)
            from scipy.ndimage import median_filter
            med = median_filter(self.current, size=3)
            self.current = (1 - strength * 0.3) * self.current + strength * 0.3 * med

        self.current = np.clip(self.current, 0, 1).astype(np.float32)
        mse_after = self._mse(self.current)
        reward = (mse_before - mse_after) * 100  # positive if improved

        self.step_count += 1
        done = self.step_count >= self.max_steps

        return self._get_state(), reward, done, {
            'mse': mse_after,
            'psnr': self._psnr(self.current),
            'deg_type': self.deg_type
        }

# Validate environment
env = VisionEnvironment()
state = env.reset(seed=0)
print(f"State shape: {state.shape}")
print(f"Initial MSE: {env._mse(env.current):.4f}")
print(f"Initial PSNR: {env._psnr(env.current):.2f} dB")

# Run a random episode
total_reward = 0
for _ in range(10):
    atype = np.random.randint(4)
    aparam = np.random.uniform(0.1, 0.9)
    state, r, done, info = env.step(atype, aparam)
    total_reward += r
print(f"After 10 random actions — MSE: {info['mse']:.4f}, PSNR: {info['psnr']:.2f} dB, Total reward: {total_reward:.4f}")

### 4.2 Neural Network Architecture

We implement three networks:
1. **Shared feature extractor** $\phi_\xi(s)$: CNN that maps images to embeddings
2. **Manager network** $\pi_{high}(o|\phi(s))$: selects which option/task to execute
3. **Worker network** $\pi_{low}(a|\phi(s), o)$: executes goal-conditioned actions

Plus termination networks $\beta_{\varphi_o}(s)$ for each option.

In [ ]:
class FeatureExtractor(nn.Module):
    """Shared CNN backbone: image -> embedding."""

    def __init__(self, img_size=32, embed_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, embed_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(0).unsqueeze(0)
        elif x.dim() == 3:
            x = x.unsqueeze(1)
        return self.net(x)


class ManagerNetwork(nn.Module):
    """High-level policy: selects option (task type)."""

    def __init__(self, embed_dim=64, n_options=4):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(embed_dim, 32),
            nn.ReLU(),
            nn.Linear(32, n_options),
        )
        self.value = nn.Sequential(
            nn.Linear(embed_dim, 32),
            nn.ReLU(),
            nn.Linear(32, n_options),
        )

    def forward(self, embedding):
        logits = self.policy(embedding)
        q_values = self.value(embedding)
        return logits, q_values


class WorkerNetwork(nn.Module):
    """Low-level goal-conditioned policy: outputs action parameters."""

    def __init__(self, embed_dim=64, n_options=4):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(embed_dim + n_options, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.action_mean = nn.Linear(32, 1)
        self.action_logstd = nn.Parameter(torch.zeros(1))
        self.value_head = nn.Linear(32, 1)

    def forward(self, embedding, option_onehot):
        x = torch.cat([embedding, option_onehot], dim=-1)
        h = self.policy(x)
        mean = torch.sigmoid(self.action_mean(h))
        std = torch.exp(self.action_logstd).expand_as(mean)
        value = self.value_head(h)
        return mean, std, value


class TerminationNetwork(nn.Module):
    """Termination function beta_o(s) for each option."""

    def __init__(self, embed_dim=64, n_options=4):
        super().__init__()
        self.nets = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, 16),
                nn.ReLU(),
                nn.Linear(16, 1),
                nn.Sigmoid(),
            ) for _ in range(n_options)
        ])

    def forward(self, embedding, option_idx):
        return self.nets[option_idx](embedding)


# Instantiate
N_OPTIONS = 4
EMBED_DIM = 64

feature_ext = FeatureExtractor(embed_dim=EMBED_DIM).to(device)
manager = ManagerNetwork(EMBED_DIM, N_OPTIONS).to(device)
worker = WorkerNetwork(EMBED_DIM, N_OPTIONS).to(device)
termination = TerminationNetwork(EMBED_DIM, N_OPTIONS).to(device)

# Count parameters
total_params = sum(p.numel() for m in [feature_ext, manager, worker, termination] for p in m.parameters())
print(f"Total parameters: {total_params:,}")
for name, model in [('FeatureExtractor', feature_ext), ('Manager', manager),
                     ('Worker', worker), ('Termination', termination)]:
    n = sum(p.numel() for p in model.parameters())
    print(f"  {name}: {n:,}")

### 4.3 Option-Critic Training Loop

We train all components jointly using the Option-Critic algorithm:

1. **Critic update**: TD error on $Q_\Omega$
2. **Intra-option policy gradient**: update Worker
3. **Termination gradient**: update $\beta_o$
4. **Policy over options**: $\epsilon$-greedy over $Q_\Omega$

In [ ]:
Transition = namedtuple('Transition', ['state', 'option', 'action_param', 'reward',
                                        'next_state', 'done', 'log_prob', 'beta'])

class HierarchicalRLTrainer:
    """Option-Critic style training for hierarchical vision RL."""

    def __init__(self, lr=3e-4, gamma=0.99, epsilon_start=0.8, epsilon_end=0.05,
                 epsilon_decay=200, termination_reg=0.01):
        self.gamma = gamma
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.termination_reg = termination_reg

        self.all_params = (
            list(feature_ext.parameters()) +
            list(manager.parameters()) +
            list(worker.parameters()) +
            list(termination.parameters())
        )
        self.optimizer = optim.Adam(self.all_params, lr=lr)
        self.episode_count = 0

    def get_epsilon(self):
        return self.epsilon_end + (self.epsilon_start - self.epsilon_end) * \
               np.exp(-self.episode_count / self.epsilon_decay)

    def select_option(self, embedding):
        """Epsilon-greedy option selection."""
        eps = self.get_epsilon()
        logits, q_values = manager(embedding)
        if np.random.random() < eps:
            return np.random.randint(N_OPTIONS)
        return q_values.argmax(dim=-1).item()

    def select_action(self, embedding, option_idx):
        """Sample action from Worker conditioned on option."""
        onehot = torch.zeros(1, N_OPTIONS, device=device)
        onehot[0, option_idx] = 1.0
        mean, std, value = worker(embedding, onehot)
        dist = torch.distributions.Normal(mean, std + 1e-6)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return torch.clamp(action, 0, 1).item(), log_prob, value

    def compute_loss(self, transitions):
        """Compute combined Option-Critic loss."""
        critic_loss = 0.0
        policy_loss = 0.0
        term_loss = 0.0

        for i, t in enumerate(transitions):
            state_t = torch.FloatTensor(t.state).to(device)
            next_state_t = torch.FloatTensor(t.next_state).to(device)

            emb = feature_ext(state_t)
            emb_next = feature_ext(next_state_t)

            _, q_vals = manager(emb)
            _, q_vals_next = manager(emb_next)

            q_o = q_vals[0, t.option]

            beta_next = termination(emb_next, t.option)
            u_next = (1 - beta_next) * q_vals_next[0, t.option] + \
                     beta_next * q_vals_next.max(dim=-1)[0]

            target = t.reward + self.gamma * u_next.detach() * (1 - float(t.done))
            critic_loss += F.mse_loss(q_o, target.squeeze())

            advantage = (target - q_o).detach()
            policy_loss += -t.log_prob * advantage

            if not t.done:
                a_omega = q_vals_next[0, t.option] - q_vals_next.max(dim=-1)[0]
                term_loss += beta_next.squeeze() * a_omega.detach() + \
                             self.termination_reg * beta_next.squeeze()

        n = max(len(transitions), 1)
        return (critic_loss + policy_loss + term_loss) / n

    def train_episode(self, env):
        """Run one episode and update."""
        state = env.reset(seed=np.random.randint(10000))
        transitions = []
        total_reward = 0.0
        option_counts = np.zeros(N_OPTIONS)

        feature_ext.train(); manager.train(); worker.train(); termination.train()

        state_t = torch.FloatTensor(state).to(device)
        emb = feature_ext(state_t)
        current_option = self.select_option(emb)

        done = False
        while not done:
            state_t = torch.FloatTensor(state).to(device)
            emb = feature_ext(state_t)

            beta_val = termination(emb, current_option).item()
            if np.random.random() < beta_val:
                current_option = self.select_option(emb)

            action_param, log_prob, value = self.select_action(emb, current_option)
            option_counts[current_option] += 1

            next_state, reward, done, info = env.step(current_option, action_param)
            total_reward += reward

            transitions.append(Transition(
                state=state, option=current_option, action_param=action_param,
                reward=reward, next_state=next_state, done=done,
                log_prob=log_prob, beta=beta_val
            ))
            state = next_state

        # Update
        self.optimizer.zero_grad()
        loss = self.compute_loss(transitions)
        loss.backward()
        nn.utils.clip_grad_norm_(self.all_params, 1.0)
        self.optimizer.step()

        self.episode_count += 1
        return {
            'reward': total_reward,
            'loss': loss.item(),
            'psnr': info['psnr'],
            'mse': info['mse'],
            'deg_type': info['deg_type'],
            'option_counts': option_counts,
            'epsilon': self.get_epsilon(),
        }

print("✅ HierarchicalRLTrainer defined.")

In [ ]:
# Train the hierarchical agent
env = VisionEnvironment(img_size=32, max_steps=10)
trainer = HierarchicalRLTrainer(lr=3e-4, gamma=0.99)

N_EPISODES = 600
history = {'reward': [], 'loss': [], 'psnr': [], 'mse': [],
           'option_usage': [], 'deg_type': [], 'epsilon': []}

print("Training Hierarchical RL Agent...")
for ep in range(N_EPISODES):
    result = trainer.train_episode(env)
    for k in ['reward', 'loss', 'psnr', 'mse', 'deg_type', 'epsilon']:
        history[k].append(result[k])
    history['option_usage'].append(result['option_counts'])

    if (ep + 1) % 100 == 0:
        avg_r = np.mean(history['reward'][-100:])
        avg_psnr = np.mean(history['psnr'][-100:])
        print(f"  Episode {ep+1:4d}/{N_EPISODES} | "
              f"Avg Reward: {avg_r:+.3f} | Avg PSNR: {avg_psnr:.2f} dB | "
              f"ε: {result['epsilon']:.3f}")

print("✅ Training complete!")

In [ ]:
# Training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def smooth(data, window=30):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Reward
ax = axes[0, 0]
ax.plot(smooth(history['reward']), color='#2ecc71', lw=1.5)
ax.fill_between(range(len(smooth(history['reward']))),
                smooth(history['reward']) - 0.3,
                smooth(history['reward']) + 0.3,
                alpha=0.2, color='#2ecc71')
ax.set_title('Episode Reward', fontweight='bold')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.grid(alpha=0.3)

# Loss
ax = axes[0, 1]
ax.plot(smooth(history['loss']), color='#e74c3c', lw=1.5)
ax.set_title('Training Loss', fontweight='bold')
ax.set_xlabel('Episode'); ax.set_ylabel('Loss')
ax.set_yscale('symlog', linthresh=1)
ax.grid(alpha=0.3)

# PSNR
ax = axes[1, 0]
ax.plot(smooth(history['psnr']), color='#3498db', lw=1.5)
ax.set_title('Output PSNR (dB)', fontweight='bold')
ax.set_xlabel('Episode'); ax.set_ylabel('PSNR (dB)')
ax.grid(alpha=0.3)

# Epsilon decay
ax = axes[1, 1]
ax.plot(history['epsilon'], color='#9b59b6', lw=1.5)
ax.set_title('Exploration Rate ε', fontweight='bold')
ax.set_xlabel('Episode'); ax.set_ylabel('ε')
ax.grid(alpha=0.3)

fig.suptitle('Hierarchical RL Training Curves', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: Training curves showing reward, loss, PSNR improvement, and exploration decay.")

### 4.4 Hierarchical Decision Visualisation

Let's see which options the trained Manager selects for each degradation type, and visualise the before/after processing results.

In [ ]:
# Evaluate on specific degradation types and visualise decisions
feature_ext.eval(); manager.eval(); worker.eval(); termination.eval()

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
col_titles = ['Clean', 'Degraded', 'Restored', 'Pixel Diff (×5)', 'Option Sequence']

for row, deg_type in enumerate(range(4)):
    env_eval = VisionEnvironment(img_size=32, max_steps=10)
    state = env_eval.reset(seed=row * 10 + 5)
    # Force degradation type
    env_eval.deg_type = deg_type
    _, fn = DEGRADATION_TYPES[deg_type]
    env_eval.current = fn(env_eval.clean).copy()
    env_eval.initial = env_eval.current.copy()
    state = env_eval._get_state()

    initial = env_eval.initial.copy()
    options_used = []

    with torch.no_grad():
        state_t = torch.FloatTensor(state).to(device)
        emb = feature_ext(state_t)
        current_option = manager(emb)[1].argmax(dim=-1).item()

        for step in range(10):
            state_t = torch.FloatTensor(state).to(device)
            emb = feature_ext(state_t)

            beta_val = termination(emb, current_option).item()
            if beta_val > 0.5:
                current_option = manager(emb)[1].argmax(dim=-1).item()

            onehot = torch.zeros(1, N_OPTIONS, device=device)
            onehot[0, current_option] = 1.0
            mean, std, _ = worker(emb, onehot)
            action_param = mean.item()

            options_used.append(current_option)
            state, r, done, info = env_eval.step(current_option, action_param)

    restored = env_eval.current
    deg_name = DEGRADATION_TYPES[deg_type][0]

    # Clean
    axes[row, 0].imshow(env_eval.clean, cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_ylabel(deg_name, fontsize=12, fontweight='bold', rotation=0, labelpad=60)

    # Degraded
    axes[row, 1].imshow(initial, cmap='gray', vmin=0, vmax=1)

    # Restored
    axes[row, 2].imshow(restored, cmap='gray', vmin=0, vmax=1)
    psnr_before = 10 * np.log10(1.0 / max(np.mean((initial - env_eval.clean)**2), 1e-10))
    psnr_after = info['psnr']
    axes[row, 2].set_title(f'{psnr_before:.1f}→{psnr_after:.1f} dB', fontsize=9, color='green')

    # Pixel difference
    diff = np.abs(restored - initial) * 5
    axes[row, 3].imshow(diff, cmap='hot', vmin=0, vmax=1)

    # Option sequence
    option_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
    for step_i, opt in enumerate(options_used):
        axes[row, 4].barh(0, 1, left=step_i, color=option_colors[opt], edgecolor='white', lw=0.5)
    axes[row, 4].set_xlim(0, 10)
    axes[row, 4].set_yticks([])
    axes[row, 4].set_xlabel('Step' if row == 3 else '')

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontweight='bold', fontsize=11)

for ax_row in axes:
    for ax in ax_row[:4]:
        ax.axis('off')

# Legend for options
handles = [plt.Rectangle((0,0),1,1, fc=option_colors[i]) for i in range(4)]
fig.legend(handles, TASK_NAMES, loc='lower center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Hierarchical Agent: Task-Specific Option Selection & Restoration',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('hierarchical_decisions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: For each degradation type, the Manager selects appropriate options.")
print("        The option sequence (rightmost column) shows temporal task switching.")

---
## 5. Analysis

### 5.1 Option Activation Patterns Across Image Types

In [ ]:
# Analyse option usage per degradation type
option_usage_by_deg = {d: np.zeros(N_OPTIONS) for d in range(4)}

feature_ext.eval(); manager.eval(); worker.eval(); termination.eval()

for trial in range(50):
    for deg_type in range(4):
        env_a = VisionEnvironment(img_size=32, max_steps=10)
        state = env_a.reset(seed=trial * 100 + deg_type)
        env_a.deg_type = deg_type
        _, fn = DEGRADATION_TYPES[deg_type]
        env_a.current = fn(env_a.clean).copy()
        state = env_a._get_state()

        with torch.no_grad():
            emb = feature_ext(torch.FloatTensor(state).to(device))
            current_option = manager(emb)[1].argmax(-1).item()

            for _ in range(10):
                emb = feature_ext(torch.FloatTensor(state).to(device))
                beta_val = termination(emb, current_option).item()
                if beta_val > 0.5:
                    current_option = manager(emb)[1].argmax(-1).item()
                option_usage_by_deg[deg_type][current_option] += 1

                onehot = torch.zeros(1, N_OPTIONS, device=device)
                onehot[0, current_option] = 1.0
                mean, _, _ = worker(emb, onehot)
                state, _, done, _ = env_a.step(current_option, mean.item())
                if done:
                    break

# Normalise
usage_matrix = np.array([option_usage_by_deg[d] for d in range(4)])
usage_norm = usage_matrix / usage_matrix.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap
ax = axes[0]
im = ax.imshow(usage_norm, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(N_OPTIONS))
ax.set_xticklabels(TASK_NAMES)
ax.set_yticks(range(4))
ax.set_yticklabels([DEGRADATION_TYPES[d][0] for d in range(4)])
ax.set_xlabel('Selected Option (Worker Task)')
ax.set_ylabel('Input Degradation Type')
ax.set_title('Option Activation Heatmap', fontweight='bold')
for i in range(4):
    for j in range(N_OPTIONS):
        ax.text(j, i, f'{usage_norm[i,j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if usage_norm[i,j] > 0.5 else 'black')
plt.colorbar(im, ax=ax, label='Proportion')

# Stacked bar chart
ax = axes[1]
bottoms = np.zeros(4)
for opt_i in range(N_OPTIONS):
    ax.bar(range(4), usage_norm[:, opt_i], bottom=bottoms,
           label=TASK_NAMES[opt_i], color=option_colors[opt_i])
    bottoms += usage_norm[:, opt_i]
ax.set_xticks(range(4))
ax.set_xticklabels([DEGRADATION_TYPES[d][0] for d in range(4)])
ax.set_ylabel('Option Usage Proportion')
ax.set_title('Option Distribution per Degradation', fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('option_activation.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: Option activation patterns show the Manager learns degradation-specific strategies.")

### 5.2 Hierarchical vs. Flat RL Comparison

We train a flat (single-level) RL agent on the same environment for comparison.

In [ ]:
class FlatAgent(nn.Module):
    """Single-level RL agent (no hierarchy) for comparison."""

    def __init__(self, img_size=32, n_actions=4):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 64), nn.ReLU(),
        )
        self.action_type_head = nn.Linear(64, n_actions)
        self.action_param_head = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )
        self.value_head = nn.Linear(64, 1)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(0).unsqueeze(0)
        elif x.dim() == 3:
            x = x.unsqueeze(1)
        feat = self.feature(x)
        action_logits = self.action_type_head(feat)
        action_param = self.action_param_head(feat)
        value = self.value_head(feat)
        return action_logits, action_param, value


def train_flat_agent(n_episodes=600, lr=3e-4, gamma=0.99):
    """Train a flat agent and return training history."""
    torch.manual_seed(42)
    flat_model = FlatAgent().to(device)
    flat_opt = optim.Adam(flat_model.parameters(), lr=lr)
    env_flat = VisionEnvironment(img_size=32, max_steps=10)
    flat_history = {'reward': [], 'psnr': [], 'loss': []}

    for ep in range(n_episodes):
        state = env_flat.reset(seed=np.random.randint(10000))
        ep_reward = 0
        log_probs = []
        values = []
        rewards_list = []

        for step in range(10):
            state_t = torch.FloatTensor(state).to(device)
            logits, param, value = flat_model(state_t)

            dist = torch.distributions.Categorical(logits=logits)
            action_type = dist.sample()
            log_prob = dist.log_prob(action_type)

            state, reward, done, info = env_flat.step(action_type.item(), param.item())
            ep_reward += reward
            log_probs.append(log_prob)
            values.append(value.squeeze())
            rewards_list.append(reward)
            if done:
                break

        # REINFORCE with baseline
        returns = []
        G = 0
        for r in reversed(rewards_list):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns).to(device)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = 0
        for lp, val, ret in zip(log_probs, values, returns):
            advantage = ret - val.detach()
            loss += -lp * advantage + 0.5 * F.mse_loss(val, ret)
        loss = loss / max(len(log_probs), 1)

        flat_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(flat_model.parameters(), 1.0)
        flat_opt.step()

        flat_history['reward'].append(ep_reward)
        flat_history['psnr'].append(info['psnr'])
        flat_history['loss'].append(loss.item())

        if (ep + 1) % 200 == 0:
            print(f"  Flat Agent — Episode {ep+1}: Avg Reward={np.mean(flat_history['reward'][-100:]):+.3f}")

    return flat_history

print("Training Flat RL Agent for comparison...")
flat_history = train_flat_agent(n_episodes=N_EPISODES)
print("✅ Flat agent training complete!")

In [ ]:
# Comparison plot: Hierarchical vs Flat
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Reward comparison
ax = axes[0]
ax.plot(smooth(history['reward'], 40), color='#e74c3c', lw=2, label='Hierarchical')
ax.plot(smooth(flat_history['reward'], 40), color='#3498db', lw=2, label='Flat', ls='--')
ax.set_title('Episode Reward', fontweight='bold')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.legend(); ax.grid(alpha=0.3)

# PSNR comparison
ax = axes[1]
ax.plot(smooth(history['psnr'], 40), color='#e74c3c', lw=2, label='Hierarchical')
ax.plot(smooth(flat_history['psnr'], 40), color='#3498db', lw=2, label='Flat', ls='--')
ax.set_title('Output PSNR (dB)', fontweight='bold')
ax.set_xlabel('Episode'); ax.set_ylabel('PSNR (dB)')
ax.legend(); ax.grid(alpha=0.3)

# Final performance bar chart
ax = axes[2]
n_final = 100
metrics = {
    'Avg Reward': [np.mean(history['reward'][-n_final:]), np.mean(flat_history['reward'][-n_final:])],
    'Avg PSNR': [np.mean(history['psnr'][-n_final:]), np.mean(flat_history['psnr'][-n_final:])],
}
x = np.arange(len(metrics))
w = 0.3
hier_vals = [v[0] for v in metrics.values()]
flat_vals = [v[1] for v in metrics.values()]
ax.bar(x - w/2, hier_vals, w, label='Hierarchical', color='#e74c3c')
ax.bar(x + w/2, flat_vals, w, label='Flat', color='#3498db')
ax.set_xticks(x)
ax.set_xticklabels(list(metrics.keys()))
ax.set_title(f'Final Performance (last {n_final} eps)', fontweight='bold')
ax.legend()
for i, (h, f) in enumerate(zip(hier_vals, flat_vals)):
    ax.text(i - w/2, h + 0.1, f'{h:.2f}', ha='center', fontsize=9, fontweight='bold')
    ax.text(i + w/2, f + 0.1, f'{f:.2f}', ha='center', fontsize=9, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

fig.suptitle('Hierarchical RL vs. Flat RL — Performance Comparison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: Hierarchical agent compared with flat baseline on reward and PSNR.")

### 5.3 Learned Termination Conditions

The termination functions $\beta_o(s)$ tell us **when** each option should stop. We visualise these across a range of image states.

In [ ]:
# Visualise termination probabilities across image processing trajectory
feature_ext.eval(); termination.eval()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for deg_idx in range(4):
    ax = axes[deg_idx // 2, deg_idx % 2]
    deg_name = DEGRADATION_TYPES[deg_idx][0]

    beta_trajectories = {opt: [] for opt in range(N_OPTIONS)}

    for trial in range(20):
        env_t = VisionEnvironment(img_size=32, max_steps=15)
        state = env_t.reset(seed=trial * 7 + deg_idx)
        env_t.deg_type = deg_idx
        _, fn = DEGRADATION_TYPES[deg_idx]
        env_t.current = fn(env_t.clean).copy()
        state = env_t._get_state()

        for step in range(10):
            with torch.no_grad():
                emb = feature_ext(torch.FloatTensor(state).to(device))
                for opt in range(N_OPTIONS):
                    b = termination(emb, opt).item()
                    beta_trajectories[opt].append((step, b))

            action_type = deg_idx  # use matched action for cleaner viz
            state, _, done, _ = env_t.step(action_type, 0.5)
            if done:
                break

    for opt in range(N_OPTIONS):
        steps_b = np.array(beta_trajectories[opt])
        # Bin by step
        mean_betas = []
        std_betas = []
        for s in range(10):
            mask = steps_b[:, 0] == s
            if mask.sum() > 0:
                mean_betas.append(steps_b[mask, 1].mean())
                std_betas.append(steps_b[mask, 1].std())
            else:
                mean_betas.append(0)
                std_betas.append(0)
        mean_betas = np.array(mean_betas)
        std_betas = np.array(std_betas)
        step_range = np.arange(10)

        ax.plot(step_range, mean_betas, 'o-', color=option_colors[opt],
                label=TASK_NAMES[opt], lw=2, markersize=4)
        ax.fill_between(step_range,
                        np.clip(mean_betas - std_betas, 0, 1),
                        np.clip(mean_betas + std_betas, 0, 1),
                        alpha=0.15, color=option_colors[opt])

    ax.set_title(f'Degradation: {deg_name}', fontweight='bold')
    ax.set_xlabel('Processing Step')
    ax.set_ylabel('$\\beta_o(s)$')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(y=0.5, color='gray', ls='--', alpha=0.4, label='Threshold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)

fig.suptitle('Learned Termination Functions $\\beta_o(s)$ Over Processing Steps',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('termination_conditions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: Termination probabilities for each option across degradation types.")
print("        High β means the option wants to terminate (switch to another).")

In [ ]:
# Comprehensive performance metrics table
print("\n" + "="*70)
print("     HIERARCHICAL RL FOR VISION — PERFORMANCE SUMMARY")
print("="*70)

# Evaluate both agents on each degradation
results_hier = {d: {'psnr_before': [], 'psnr_after': [], 'reward': []} for d in range(4)}

feature_ext.eval(); manager.eval(); worker.eval(); termination.eval()

for trial in range(30):
    for deg_type in range(4):
        env_e = VisionEnvironment(img_size=32, max_steps=10)
        state = env_e.reset(seed=trial * 50 + deg_type + 1000)
        env_e.deg_type = deg_type
        _, fn = DEGRADATION_TYPES[deg_type]
        env_e.current = fn(env_e.clean).copy()
        env_e.initial = env_e.current.copy()

        psnr_before = 10 * np.log10(1.0 / max(np.mean((env_e.initial - env_e.clean)**2), 1e-10))
        results_hier[deg_type]['psnr_before'].append(psnr_before)

        state = env_e._get_state()
        total_r = 0
        with torch.no_grad():
            emb = feature_ext(torch.FloatTensor(state).to(device))
            current_option = manager(emb)[1].argmax(-1).item()
            for _ in range(10):
                emb = feature_ext(torch.FloatTensor(state).to(device))
                beta_val = termination(emb, current_option).item()
                if beta_val > 0.5:
                    current_option = manager(emb)[1].argmax(-1).item()
                onehot = torch.zeros(1, N_OPTIONS, device=device)
                onehot[0, current_option] = 1.0
                mean, _, _ = worker(emb, onehot)
                state, r, done, info = env_e.step(current_option, mean.item())
                total_r += r
                if done:
                    break
        results_hier[deg_type]['psnr_after'].append(info['psnr'])
        results_hier[deg_type]['reward'].append(total_r)

print(f"\n{'Degradation':<15} {'PSNR Before':>12} {'PSNR After':>12} {'Δ PSNR':>10} {'Avg Reward':>12}")
print("-" * 65)
for d in range(4):
    name = DEGRADATION_TYPES[d][0]
    pb = np.mean(results_hier[d]['psnr_before'])
    pa = np.mean(results_hier[d]['psnr_after'])
    ar = np.mean(results_hier[d]['reward'])
    print(f"{name:<15} {pb:>10.2f} dB {pa:>10.2f} dB {pa-pb:>+8.2f} dB {ar:>+10.3f}")

overall_before = np.mean([np.mean(results_hier[d]['psnr_before']) for d in range(4)])
overall_after = np.mean([np.mean(results_hier[d]['psnr_after']) for d in range(4)])
overall_reward = np.mean([np.mean(results_hier[d]['reward']) for d in range(4)])
print("-" * 65)
print(f"{'Overall':<15} {overall_before:>10.2f} dB {overall_after:>10.2f} dB "
      f"{overall_after - overall_before:>+8.2f} dB {overall_reward:>+10.3f}")
print("=" * 70)

In [ ]:
# Final visualisation: option usage evolution during training
option_usage_arr = np.array(history['option_usage'])  # shape (N_EPISODES, N_OPTIONS)
window = 50

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Smoothed option usage over training
ax = axes[0]
for opt_i in range(N_OPTIONS):
    smoothed_usage = smooth(option_usage_arr[:, opt_i], window)
    ax.plot(smoothed_usage, color=option_colors[opt_i], lw=2, label=TASK_NAMES[opt_i])
ax.set_title('Option Usage Over Training', fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Steps Using Option (per episode)')
ax.legend()
ax.grid(alpha=0.3)

# Option entropy over training (diversity of option usage)
ax = axes[1]
entropies = []
for ep in range(N_EPISODES):
    total_steps = option_usage_arr[ep].sum()
    if total_steps > 0:
        probs = option_usage_arr[ep] / total_steps
        probs = probs[probs > 0]
        entropies.append(-np.sum(probs * np.log(probs + 1e-10)))
    else:
        entropies.append(0)

ax.plot(smooth(entropies, window), color='#8e44ad', lw=2)
ax.axhline(y=np.log(N_OPTIONS), color='gray', ls='--', alpha=0.5, label='Max entropy (uniform)')
ax.set_title('Option Usage Entropy Over Training', fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Entropy (nats)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('option_evolution.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure: Left — how option usage evolves during training.")
print("        Right — entropy of option selection (lower = more specialised).")

---
## 6. Summary

### Key Takeaways

| Concept | Core Idea |
|---------|----------|
| **Options Framework** | Extends MDPs with temporally extended actions $o = \langle \mathcal{I}_o, \pi_o, \beta_o \rangle$ |
| **SMDP** | Semi-Markov process $\mathcal{M}_{SMDP} = \langle \mathcal{S}, \mathcal{O}, T_o, R_o, \gamma \rangle$ models variable-duration options |
| **Option-Value Function** | $Q_\Omega(s,o)$ decomposes into continue vs. terminate via $U(o,s')$ |
| **Option-Critic** | End-to-end learning of intra-option policies, termination functions, and critic |
| **Manager–Worker** | High-level selects vision task (goal), low-level executes pixel operations |
| **Goal-Conditioned Policy** | $\pi_{low}(a \mid s, g)$ where $g = \pi_{high}(s)$ enables reusable skills |
| **Intrinsic Reward** | Workers rewarded for goal achievement: $r^{int} = -\|\phi(s') - g\|^2$ |

### What We Demonstrated

1. **Automatic task selection**: The Manager learned to match degradation types to appropriate processing options
2. **Temporal abstraction**: Options execute for variable durations, terminating when the sub-task is complete
3. **Improved performance**: Hierarchical decomposition enables more structured exploration than flat RL
4. **Interpretability**: We can inspect which option is active and when it terminates

### Connections to Advanced Topics

- **Module 11.1 (Multi-Agent Vision)**: Multiple workers can be viewed as cooperative agents
- **Module 11.3 (Meta-Learning)**: Options can be meta-learned across image domains
- **Module 11.4 (Curriculum Learning)**: Manager's option selection naturally creates a curriculum

---

*Module 11.2 — Hierarchical RL for Vision | Reinforcement Learning for Image Processing*